# V1-12: 출처 기반 해석 정책의 `NOT_FOUND` cutoff 비교

11번에서 조건부 채택한 C 정책을 고정한다. 즉, 고유 exact는 확정하고, 명시적 별칭·오타 정정은 안내와 함께 확정하며, 동명·포괄 표현은 재질문한다. 이번에는 그 어느 경로에도 해당하지 않는 fuzzy 후보를 언제 `NOT_FOUND`로 처리할지 cutoff를 비교한다.

LLM·LangChain은 사용하지 않는다.

## 1. 비교 대상

- **cutoff 없음**: fuzzy 후보가 하나라도 있으면 `NEEDS_CONFIRMATION`
- **cutoff 60/70/75/80/85**: fuzzy 최상위 점수가 cutoff보다 낮으면 `NOT_FOUND`, 같거나 높으면 `NEEDS_CONFIRMATION`

exact·별칭·오타 정정·포괄 표현은 cutoff를 거치지 않는다. 따라서 cutoff가 `삼성전쟈` 같은 명시적 정정 규칙을 차단하지 않는지와, 아직 사전에 없는 fuzzy 유효 입력을 과도하게 차단하지 않는지를 분리해 본다.

공식 원천: [OpenDART 고유번호 API 개발가이드](https://opendart.fss.or.kr/guide/detail.do?apiGrpCd=DS001&apiId=2019018), [RapidFuzz process 문서](https://rapidfuzz.github.io/RapidFuzz/Usage/process.html).

In [ ]:
import io
import os
import re
import sys
import time
import xml.etree.ElementTree as ET
import zipfile
from math import ceil
from pathlib import Path
from statistics import median

from rapidfuzz import fuzz, process


In [ ]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')

for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

api_key = os.environ.get('OPENDART_API_KEY')
if not api_key:
    raise ValueError('.env에 OPENDART_API_KEY를 설정해 주세요.')

sys.path.insert(0, str(project_root / 'src'))
from dart.client import CORP_CODE_URL, _get_bytes


In [ ]:
def load_corp_records(api_key):
    zip_bytes = _get_bytes(CORP_CODE_URL, {'crtfc_key': api_key})
    zip_stream = io.BytesIO(zip_bytes)
    if not zipfile.is_zipfile(zip_stream):
        raise RuntimeError(zip_bytes.decode('utf-8', errors='replace'))
    with zipfile.ZipFile(zip_stream) as archive:
        xml_bytes = archive.read(archive.namelist()[0])
    root = ET.fromstring(xml_bytes)
    return [{child.tag: child.text or '' for child in item} for item in root.findall('list')]

def normalize_name(value):
    return re.sub(r'[^0-9A-Z가-힣]', '', value.upper())

def build_name_index(records):
    index = {}
    for record in records:
        for name in {record['corp_name'], record['corp_eng_name']}:
            key = normalize_name(name)
            if key:
                index.setdefault(key, {})[record['corp_code']] = record
    return {key: list(by_code.values()) for key, by_code in index.items()}

corp_records = load_corp_records(api_key)
name_index = build_name_index(corp_records)
name_keys = list(name_index)
print(f'전체 법인 레코드 수: {len(corp_records):,}')
print(f'정규화 이름 키 수: {len(name_keys):,}')


## 2. C 정책과 평가 사례

fuzzy 유효 입력은 아직 별칭·오타 사전에 넣지 않은 오타 표현이다. 따라서 이 사례를 `NOT_FOUND`로 차단하면 cutoff가 미래에 추가될 수 있는 유효 표현을 놓친다. 대상 부재 입력은 현재 OpenDART 이름 인덱스에 정확 일치하지 않음을 assert로 확인한 synthetic no-match다.

In [ ]:
INTERPRETED_TARGETS = {
    normalize_name('삼성전쟈'): ('005930', '입력한 삼성전쟈는 삼성전자로 해석했습니다.'),
    normalize_name('하이닉스'): ('000660', '입력한 하이닉스는 SK하이닉스로 해석했습니다.'),
    normalize_name('하닉'): ('000660', '입력한 하닉은 SK하이닉스로 해석했습니다.'),
    normalize_name('엘지전자'): ('066570', '입력한 엘지전자는 LG전자로 해석했습니다.'),
    normalize_name('현대차'): ('005380', '입력한 현대차는 현대자동차로 해석했습니다.'),
    normalize_name('네이버'): ('035420', '입력한 네이버는 NAVER로 해석했습니다.'),
}
AMBIGUOUS_INPUTS = {normalize_name(value) for value in ['삼전', '엘지', '현대', '삼성']}

TEST_CASES = [
    {'질문 기업명': '삼성전자', '유형': 'exact', '대상 존재': True, '기대 상태': 'RESOLVED'},
    {'질문 기업명': '삼성전쟈', '유형': 'interpreted typo', '대상 존재': True, '기대 상태': 'RESOLVED_INTERPRETED'},
    {'질문 기업명': '하이닉스', '유형': 'interpreted alias', '대상 존재': True, '기대 상태': 'RESOLVED_INTERPRETED'},
    {'질문 기업명': '하닉', '유형': 'interpreted alias', '대상 존재': True, '기대 상태': 'RESOLVED_INTERPRETED'},
    {'질문 기업명': '엘지전자', '유형': 'interpreted alias', '대상 존재': True, '기대 상태': 'RESOLVED_INTERPRETED'},
    {'질문 기업명': '현대차', '유형': 'interpreted alias', '대상 존재': True, '기대 상태': 'RESOLVED_INTERPRETED'},
    {'질문 기업명': '네이버', '유형': 'interpreted alias', '대상 존재': True, '기대 상태': 'RESOLVED_INTERPRETED'},
    {'질문 기업명': '삼전', '유형': 'ambiguous', '대상 존재': True, '기대 상태': 'NEEDS_CONFIRMATION'},
    {'질문 기업명': '엘지', '유형': 'ambiguous', '대상 존재': True, '기대 상태': 'NEEDS_CONFIRMATION'},
    {'질문 기업명': '현대', '유형': 'ambiguous', '대상 존재': True, '기대 상태': 'NEEDS_CONFIRMATION'},
    {'질문 기업명': '삼성', '유형': 'ambiguous', '대상 존재': True, '기대 상태': 'NEEDS_CONFIRMATION'},
    {'질문 기업명': '삼성전저', '유형': 'fuzzy valid typo', '대상 존재': True, '기대 상태': None},
    {'질문 기업명': '카카오오', '유형': 'fuzzy valid typo', '대상 존재': True, '기대 상태': None},
    {'질문 기업명': '현대자동챠', '유형': 'fuzzy valid typo', '대상 존재': True, '기대 상태': None},
    {'질문 기업명': 'POSCO홀딩수', '유형': 'fuzzy valid typo', '대상 존재': True, '기대 상태': None},
    {'질문 기업명': '감자우주물산2026', '유형': 'synthetic no-match', '대상 존재': False, '기대 상태': None},
    {'질문 기업명': 'BLUEORBITVENTURES2049', '유형': 'synthetic no-match', '대상 존재': False, '기대 상태': None},
    {'질문 기업명': 'QXZ테스트법인9999', '유형': 'synthetic no-match', '대상 존재': False, '기대 상태': None},
    {'질문 기업명': '삼성전자파트너스', '유형': 'synthetic no-match', '대상 존재': False, '기대 상태': None},
    {'질문 기업명': 'SK하이닉스솔루션', '유형': 'synthetic no-match', '대상 존재': False, '기대 상태': None},
    {'질문 기업명': '완전없는회사마루2026', '유형': 'synthetic no-match', '대상 존재': False, '기대 상태': None},
]

for case in TEST_CASES:
    key = normalize_name(case['질문 기업명'])
    if not case['대상 존재']:
        assert key not in name_index
    if case['유형'] == 'fuzzy valid typo':
        assert key not in name_index
        assert key not in INTERPRETED_TARGETS
        assert key not in AMBIGUOUS_INPUTS

print(f'전체 사례: {len(TEST_CASES)}개')
print(f'fuzzy 유효 입력: {sum(case["유형"] == "fuzzy valid typo" for case in TEST_CASES)}개')
print(f'대상 부재 입력: {sum(not case["대상 존재"] for case in TEST_CASES)}개')


In [ ]:
FUZZY_KEY_LIMIT = 50
FUZZY_RESULT_LIMIT = 5

def exact_records(query):
    return name_index.get(normalize_name(query), [])

def fuzzy_candidates(query):
    scored_keys = process.extract(
        normalize_name(query), name_keys, scorer=fuzz.ratio, limit=FUZZY_KEY_LIMIT
    )
    best_by_code = {}
    for key, score, _ in scored_keys:
        for record in name_index[key]:
            current = best_by_code.get(record['corp_code'])
            if current is None or score > current[1]:
                best_by_code[record['corp_code']] = (record, score)
    return sorted(
        best_by_code.values(),
        key=lambda item: (item[1], bool(item[0]['stock_code'].strip()), item[0]['modify_date']),
        reverse=True,
    )[:FUZZY_RESULT_LIMIT]

def c_policy_status(query, cutoff):
    key = normalize_name(query)
    if key in AMBIGUOUS_INPUTS:
        return 'NEEDS_CONFIRMATION', 'ambiguous_input', None
    if key in INTERPRETED_TARGETS:
        return 'RESOLVED_INTERPRETED', 'interpreted', None
    exact = exact_records(query)
    if len(exact) == 1:
        return 'RESOLVED', 'exact', None
    if len(exact) > 1:
        return 'NEEDS_CONFIRMATION', 'multiple_exact', None

    candidates = fuzzy_candidates(query)
    top_score = candidates[0][1] if candidates else None
    if not candidates or (cutoff is not None and top_score < cutoff):
        return 'NOT_FOUND', 'fuzzy_cutoff', top_score
    return 'NEEDS_CONFIRMATION', 'fuzzy_confirmation', top_score


## 3. cutoff grid 평가

`fuzzy valid typo`에 대한 Fuzzy-path False Not-found Rate는 cutoff가 미등록이지만 유효한 표현을 얼마나 차단하는지 직접 보여 준다. 이 값이 올라가면 Not-found Recall이 좋아져도 정확도 우선 정책에서는 해당 cutoff를 채택하지 않는다.

In [ ]:
CUTOFF_CANDIDATES = [None, 60.0, 70.0, 75.0, 80.0, 85.0]
WARMUP_ITERATIONS = 2
MEASURE_ITERATIONS = 10

grid_rows = {}
grid_summaries = []
for cutoff in CUTOFF_CANDIDATES:
    label = 'A. cutoff 없음' if cutoff is None else f'cutoff {cutoff:.0f}'
    rows = []
    latencies = []
    for case in TEST_CASES:
        for _ in range(WARMUP_ITERATIONS):
            c_policy_status(case['질문 기업명'], cutoff)
        measurements = []
        for _ in range(MEASURE_ITERATIONS):
            started_at = time.perf_counter()
            status, route, top_score = c_policy_status(case['질문 기업명'], cutoff)
            measurements.append((time.perf_counter() - started_at) * 1_000)
        rows.append({**case, 'cutoff': cutoff, '상태': status, '경로': route, '최고 점수': top_score})
        latencies.extend(measurements)

    missing_rows = [row for row in rows if not row['대상 존재']]
    fuzzy_valid_rows = [row for row in rows if row['유형'] == 'fuzzy valid typo']
    protected_rows = [row for row in rows if row['기대 상태'] is not None]
    grid_rows[label] = rows
    grid_summaries.append({
        '정책': label,
        'Not-found Recall': sum(row['상태'] == 'NOT_FOUND' for row in missing_rows) / len(missing_rows),
        'Fuzzy-path False Not-found Rate': sum(row['상태'] == 'NOT_FOUND' for row in fuzzy_valid_rows) / len(fuzzy_valid_rows),
        'Policy Preservation Accuracy': sum(
            row['상태'] == row['기대 상태'] for row in protected_rows
        ) / len(protected_rows),
        'Clarification Rate': sum(row['상태'] == 'NEEDS_CONFIRMATION' for row in rows) / len(rows),
        'p50 지연시간(ms)': median(latencies),
        'p95 지연시간(ms)': sorted(latencies)[ceil(len(latencies) * 0.95) - 1],
    })

for summary in grid_summaries:
    print(summary)


## 4. 사례별 cutoff 영향을 확인한다

특히 fuzzy 유효 입력과 대상 부재 입력의 최고 점수·상태를 같이 본다. cutoff가 같거나 높은 점수에서는 `NEEDS_CONFIRMATION`을 유지하고, 낮은 점수에서만 `NOT_FOUND`로 바뀌는지 확인한다.

In [ ]:
FOCUS_INPUTS = [
    '삼성전쟈', '하닉', '삼전', '삼성전저', '카카오오', '현대자동챠', 'POSCO홀딩수',
    '감자우주물산2026', 'BLUEORBITVENTURES2049', '삼성전자파트너스', 'SK하이닉스솔루션',
]

for query in FOCUS_INPUTS:
    print(f'\n=== {query} ===')
    for label, rows in grid_rows.items():
        row = next(row for row in rows if row['질문 기업명'] == query)
        print({'정책': label, '유형': row['유형'], '상태': row['상태'], '경로': row['경로'], '최고 점수': row['최고 점수']})


## 5. 해석 기준

- Not-found Recall이 올라가도 Fuzzy-path False Not-found Rate가 0보다 커지면, 해당 cutoff는 유효한 미등록 표현을 차단하므로 정확도 우선 기준에서 미채택이다.
- Policy Preservation Accuracy가 1.0이 아니면 cutoff가 우회해야 하는 exact·해석·포괄 표현 경로까지 침범했거나, 기존 C 정책과 다르게 동작한 구현 오류다.
- 여러 cutoff가 같은 결과를 내면 현재 개발 사례만으로 그 값을 구분할 수 없으므로 최종 운영값을 임의로 정하지 않는다.
- 이번 synthetic no-match는 실제 사용자 입력 전체를 대표하지 않는다. 선택한 cutoff는 초기 기준일 뿐이며 운영 로그로 재평가해야 한다.

이 notebook은 최종 pipeline이나 `src/` 코드를 변경하지 않는다.